# Node Wars: The Link Awakens — Unified Reproducibility Notebook**Auditing Embedding-Space Stability in Drug-Drug Interaction GNNs**Jalen Stephens & Gabriel Mieses · Applied Machine Learning · Spring 2026This single notebook reproduces every figure and number that appears in ourblog post (`blog/index.html`). It is structured so a fresh Colab session canupload the `.ipynb`, click **Runtime → Run all**, and watch every plot render.**What this notebook does, end to end:**1. Downloads the four Stanford BioSNAP datasets.2. Builds the heterogeneous knowledge graph (drug, gene, disease nodes; 4 +   reverse edge types).3. Computes the parameter-free common-neighbor heuristic baseline.4. Trains the structure-only GCN and the multi-relational R-GCN.5. Evaluates both models (AUROC, AUPRC, MRR, Hits@K) and plots ROC, PR,   training curves.6. Extracts the learned drug embeddings and runs the stability audit   (Gaussian noise at 6 σ levels, dimensional dropout at 5 rates, 10 trials   each).7. Trains 3 edge-type ablation variants of the R-GCN and a cold-start variant.8. Reproduces every figure used in the blog (model bars, Hits@K, ROC,   training curves, Stability Score, Spearman ρ, |Δp|, top-K Jaccard,   dropout comparisons, ablation, cold-start, scatter, and the SS components).**Runtime:** if cached checkpoints and result JSONs are present in the repo(default for our committed GitHub repo), the notebook completes in ~2 min onColab CPU. With `RUN_FROM_SCRATCH = True` (set in the next cell) it retrainsall models, which takes ≈30 min on Colab CPU and ≈5 min on a T4 GPU.**Notebook → blog mapping:** every figure rendered here corresponds to asection of `blog/index.html`. The section titles below mirror the §-numberedsections of the post.

## § Setup — install dependencies and pull cached artifactsRuns the same on Google Colab and a local Jupyter kernel. On Colab it clonesthe project's GitHub repo so the cached BioSNAP downloads, model checkpoints,and pre-computed result JSONs are available without re-training.

In [ ]:
# ── Toggle: re-train every model from scratch (slow), or use cached ckpts ──
RUN_FROM_SCRATCH = False  # set to True to retrain GCN, RGCN, ablations, and cold-start models

# Reasonable defaults that reproduce the blog numbers; raise if you need more.
EPOCHS_GCN  = 100   # blog: 62 epochs (early-stopped from 200)
EPOCHS_RGCN = 500   # blog: 500 epochs
NUM_TRIALS  = 10    # stability trials per perturbation level (blog setting)
SEED        = 42

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

# Clone the project repo on Colab so cached checkpoints and processed data are available.
REPO_URL = 'https://github.com/Jalen-Stephens/ML-Project.git'
if IN_COLAB and not os.path.isdir('ML-Project'):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL])

if IN_COLAB:
    os.chdir('ML-Project')

print('Working directory:', os.getcwd())
print('Repo contents:', sorted(os.listdir('.')))

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────
# Colab ships with PyTorch and most scientific-Python libs. We add torch_geometric
# (CPU build) and pyyaml/seaborn if missing.
import importlib

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pip_name or pkg])

_ensure('torch_geometric', 'torch-geometric')
_ensure('seaborn')
_ensure('yaml', 'pyyaml')
_ensure('tqdm')
_ensure('requests')

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## § Inlined source libraryThe original repo splits these helpers across `src/data_loading.py`,`src/preprocessing.py`, `src/graph_builder.py`, `src/models.py`,`src/training.py`, `src/evaluation.py`, `src/perturbation.py`,`src/stability.py`. We inline them here so the notebook is self-contained anddoes not depend on the package layout.

In [ ]:
import os, json, gzip, pickle, random, time, math
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import requests

import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from scipy.stats import spearmanr
from scipy.sparse import coo_matrix

from torch_geometric.nn import GCNConv, RGCNConv
from torch_geometric.data import HeteroData

# Repro
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Paths
PROJECT_ROOT  = os.getcwd()
DATA_RAW      = os.path.join(PROJECT_ROOT, 'data', 'raw')
DATA_PROC     = os.path.join(PROJECT_ROOT, 'data', 'processed')
DATA_SPLITS   = os.path.join(PROJECT_ROOT, 'data', 'splits')
RESULTS_DIR   = os.path.join(PROJECT_ROOT, 'experiments', 'results')
CKPT_DIR      = os.path.join(PROJECT_ROOT, 'experiments', 'checkpoints')
BLOG_ASSETS   = os.path.join(PROJECT_ROOT, 'blog', 'assets')
for d in [DATA_RAW, DATA_PROC, DATA_SPLITS, RESULTS_DIR, CKPT_DIR, BLOG_ASSETS]:
    os.makedirs(d, exist_ok=True)

if RUN_FROM_SCRATCH:
    scratch_files = [
        # Processed graph cache
        os.path.join(DATA_PROC, 'id_maps.pkl'),
        os.path.join(DATA_PROC, 'stats.pkl'),
        os.path.join(DATA_PROC, 'edges_drug_drug.npy'),
        os.path.join(DATA_PROC, 'edges_drug_gene.npy'),
        os.path.join(DATA_PROC, 'edges_disease_gene.npy'),
        os.path.join(DATA_PROC, 'edges_disease_drug.npy'),
        # Split caches
        os.path.join(DATA_SPLITS, 'train_edges.pt'),
        os.path.join(DATA_SPLITS, 'val_edges.pt'),
        os.path.join(DATA_SPLITS, 'test_edges.pt'),
        os.path.join(DATA_SPLITS, 'train_edges_ud.pt'),
        os.path.join(DATA_SPLITS, 'cold_start_drugs.pt'),
        os.path.join(DATA_SPLITS, 'cold_edges.pt'),
        os.path.join(DATA_SPLITS, 'cs_train_edges.pt'),
        os.path.join(DATA_SPLITS, 'cs_val_edges.pt'),
        os.path.join(DATA_SPLITS, 'cs_test_warm_edges.pt'),
        # Result caches
        os.path.join(RESULTS_DIR, 'gcn_history.json'),
        os.path.join(RESULTS_DIR, 'rgcn_history.json'),
        os.path.join(RESULTS_DIR, 'ablation_results.json'),
        os.path.join(RESULTS_DIR, 'coldstart_results.json'),
        os.path.join(RESULTS_DIR, 'stability_results.json'),
        os.path.join(RESULTS_DIR, 'gcn_base_s42_metrics.json'),
        os.path.join(RESULTS_DIR, 'rgcn_base_s42_metrics.json'),
        os.path.join(RESULTS_DIR, 'summary.csv'),
        # Model checkpoints
        os.path.join(CKPT_DIR, f'gcn_base_s{SEED}.pt'),
        os.path.join(CKPT_DIR, f'rgcn_base_s{SEED}.pt'),
    ]
    removed = 0
    for fp in scratch_files:
        if os.path.exists(fp):
            os.remove(fp)
            removed += 1
    print(f'[scratch] removed {removed} cached artifacts')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Plotting defaults — match the blog look-and-feel
matplotlib.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11, 'legend.fontsize': 10,
    'figure.figsize': (9, 5),
})
COLORS = {'heuristic': '#06b6d4', 'gcn': '#3b82f6', 'rgcn': '#8b5cf6'}
COLORS_LITE = {'heuristic': '#67e8f9', 'gcn': '#93c5fd', 'rgcn': '#c4b5fd'}

In [ ]:
# ── Dataset registry & loaders ───────────────────────────────────────
DATASETS = {
    'ChCh-Miner':    {'url': 'https://snap.stanford.edu/biodata/datasets/10001/files/ChCh-Miner_durgbank-chem-chem.tsv.gz',
                      'filename': 'ChCh-Miner_durgbank-chem-chem.tsv.gz'},
    'ChG-Miner':     {'url': 'https://snap.stanford.edu/biodata/datasets/10002/files/ChG-Miner_miner-chem-gene.tsv.gz',
                      'filename': 'ChG-Miner_miner-chem-gene.tsv.gz'},
    'DG-AssocMiner': {'url': 'https://snap.stanford.edu/biodata/datasets/10012/files/DG-AssocMiner_miner-disease-gene.tsv.gz',
                      'filename': 'DG-AssocMiner_miner-disease-gene.tsv.gz'},
    'DCh-Miner':     {'url': 'https://snap.stanford.edu/biodata/datasets/10004/files/DCh-Miner_miner-disease-chemical.tsv.gz',
                      'filename': 'DCh-Miner_miner-disease-chemical.tsv.gz'},
}

def download_dataset(name, force=False):
    info = DATASETS[name]
    fp = os.path.join(DATA_RAW, info['filename'])
    if os.path.exists(fp) and not force:
        return fp
    print(f'[download] {name}')
    r = requests.get(info['url'], stream=True, timeout=120); r.raise_for_status()
    with open(fp, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1<<20):
            f.write(chunk)
    return fp

def load_tsv(name):
    fp = download_dataset(name)
    if name == 'ChCh-Miner':
        df = pd.read_csv(fp, sep='\t', compression='gzip', header=None, names=['drug1','drug2'])
    elif name == 'ChG-Miner':
        df = pd.read_csv(fp, sep='\t', compression='gzip', comment='#', header=None, names=['drug','gene'])
    elif name == 'DG-AssocMiner':
        df = pd.read_csv(fp, sep='\t', compression='gzip', comment='#', header=None,
                         names=['disease','disease_name','gene'])
        df['gene'] = 'ENTREZ:' + df['gene'].astype(str).str.strip()
        df['disease_name'] = df['disease_name'].str.strip('" ')
    elif name == 'DCh-Miner':
        df = pd.read_csv(fp, sep='\t', compression='gzip', comment='#', header=None, names=['disease','drug'])
    df = df.dropna().drop_duplicates()
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip()
    return df

def load_all():
    return {name: load_tsv(name) for name in DATASETS}

In [ ]:
# ── Preprocessing & graph builders ───────────────────────────────────
def prefix_chg_genes(df):
    df = df.copy()
    mask = ~df['gene'].str.startswith('UNIPROT:')
    df.loc[mask, 'gene'] = 'UNIPROT:' + df.loc[mask, 'gene']
    return df

def build_id_maps(ddi, dtg, dga, dda=None):
    drug_ids = set(ddi['drug1'].unique()) | set(ddi['drug2'].unique())
    gene_ids = set(dtg['gene'].unique()) | set(dga['gene'].unique())
    disease_ids = set(dga['disease'].unique())
    if dda is not None:
        disease_ids |= set(dda['disease'].unique())
    drug_map    = {x:i for i,x in enumerate(sorted(drug_ids))}
    gene_map    = {x:i for i,x in enumerate(sorted(gene_ids))}
    disease_map = {x:i for i,x in enumerate(sorted(disease_ids))}
    stats = {
        'num_drugs': len(drug_map), 'num_genes': len(gene_map),
        'num_diseases': len(disease_map),
        'drugs_with_gene_targets': len(set(dtg['drug'].unique()) & drug_ids),
    }
    if dda is not None:
        stats['drugs_with_disease_assoc'] = len(set(dda['drug'].unique()) & drug_ids)
    return {'drug': drug_map, 'gene': gene_map, 'disease': disease_map}, stats

def build_edge_index(df, src_col, dst_col, src_map, dst_map):
    mask = df[src_col].isin(src_map) & df[dst_col].isin(dst_map)
    f = df[mask]
    return np.stack([f[src_col].map(src_map).values.astype(np.int64),
                     f[dst_col].map(dst_map).values.astype(np.int64)], axis=0)

def build_all_edges(ddi, dtg, dga, id_maps, dda=None):
    drug_map, gene_map, disease_map = id_maps['drug'], id_maps['gene'], id_maps['disease']
    ddi_ei = build_edge_index(ddi, 'drug1', 'drug2', drug_map, drug_map)
    ddi_ei = np.unique(np.concatenate([ddi_ei, np.stack([ddi_ei[1], ddi_ei[0]])], axis=1), axis=1)
    edges = {
        'drug_drug':    ddi_ei,
        'drug_gene':    build_edge_index(dtg, 'drug', 'gene', drug_map, gene_map),
        'disease_gene': build_edge_index(dga, 'disease', 'gene', disease_map, gene_map),
    }
    if dda is not None:
        edges['disease_drug'] = build_edge_index(dda, 'disease', 'drug', disease_map, drug_map)
    return edges

def save_processed(id_maps, edges, stats):
    with open(os.path.join(DATA_PROC, 'id_maps.pkl'), 'wb') as f: pickle.dump(id_maps, f)
    for n, ei in edges.items():
        np.save(os.path.join(DATA_PROC, f'edges_{n}.npy'), ei)
    with open(os.path.join(DATA_PROC, 'stats.pkl'), 'wb') as f: pickle.dump(stats, f)

def load_processed():
    with open(os.path.join(DATA_PROC, 'id_maps.pkl'), 'rb') as f: id_maps = pickle.load(f)
    edges = {fn[6:-4]: np.load(os.path.join(DATA_PROC, fn))
             for fn in os.listdir(DATA_PROC) if fn.startswith('edges_') and fn.endswith('.npy')}
    with open(os.path.join(DATA_PROC, 'stats.pkl'), 'rb') as f: stats = pickle.load(f)
    return id_maps, edges, stats

def build_hetero_data(id_maps, edges):
    d = HeteroData()
    d['drug'].num_nodes    = len(id_maps['drug'])
    d['gene'].num_nodes    = len(id_maps['gene'])
    d['disease'].num_nodes = len(id_maps['disease'])
    if 'drug_drug' in edges:
        d['drug', 'interacts', 'drug'].edge_index = torch.from_numpy(edges['drug_drug']).long()
    if 'drug_gene' in edges:
        ei = torch.from_numpy(edges['drug_gene']).long()
        d['drug', 'targets', 'gene'].edge_index = ei
        d['gene', 'targeted_by', 'drug'].edge_index = torch.stack([ei[1], ei[0]])
    if 'disease_gene' in edges:
        ei = torch.from_numpy(edges['disease_gene']).long()
        d['disease', 'associated_with', 'gene'].edge_index = ei
        d['gene', 'associated_with', 'disease'].edge_index = torch.stack([ei[1], ei[0]])
    if 'disease_drug' in edges:
        ei = torch.from_numpy(edges['disease_drug']).long()
        d['disease', 'treated_by', 'drug'].edge_index = ei
        d['drug', 'treats', 'disease'].edge_index = torch.stack([ei[1], ei[0]])
    return d

def build_homo_data(id_maps, edges):
    return torch.from_numpy(edges['drug_drug']).long(), len(id_maps['drug'])

In [ ]:
# ── Models ───────────────────────────────────────────────────────────
class LinkDecoder(nn.Module):
    def forward(self, z_src, z_dst):
        return (z_src * z_dst).sum(dim=-1)

class GCNLinkPredictor(nn.Module):
    def __init__(self, num_nodes, embed_dim=64, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, embed_dim)
        self.conv1 = GCNConv(embed_dim, embed_dim)
        self.conv2 = GCNConv(embed_dim, embed_dim)
        self.decoder = LinkDecoder()
        self.dropout = dropout
        nn.init.xavier_uniform_(self.embedding.weight)
    def encode(self, edge_index):
        x = self.conv1(self.embedding.weight, edge_index)
        x = F.relu(x); x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)
    def decode(self, z, eli):
        return self.decoder(z[eli[0]], z[eli[1]])
    def forward(self, edge_index, eli):
        return self.decode(self.encode(edge_index), eli)

class RGCNLinkPredictor(nn.Module):
    def __init__(self, num_nodes_dict, embed_dim=64, num_relations=7, num_bases=2, dropout=0.3):
        super().__init__()
        self.node_types = sorted(num_nodes_dict.keys())
        self.num_nodes_dict = num_nodes_dict
        self.embeddings = nn.ModuleDict({
            n: nn.Embedding(num_nodes_dict[n], embed_dim) for n in self.node_types
        })
        self.conv1 = RGCNConv(embed_dim, embed_dim, num_relations=num_relations, num_bases=num_bases)
        self.conv2 = RGCNConv(embed_dim, embed_dim, num_relations=num_relations, num_bases=num_bases)
        self.decoder = LinkDecoder()
        self.dropout = dropout
        offsets = {}; off = 0
        for n in self.node_types:
            offsets[n] = off; off += num_nodes_dict[n]
        self._offsets = offsets
        for emb in self.embeddings.values():
            nn.init.xavier_uniform_(emb.weight)
    @property
    def drug_offset(self): return self._offsets['drug']
    @property
    def num_drugs(self):   return self.num_nodes_dict['drug']
    def get_initial_embeddings(self):
        return torch.cat([self.embeddings[n].weight for n in self.node_types], dim=0)
    def encode(self, edge_index, edge_type):
        x = self.get_initial_embeddings()
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x); x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index, edge_type)
    def get_drug_embeddings(self, z):
        return z[self.drug_offset : self.drug_offset + self.num_drugs]
    def decode(self, z, eli):
        zd = self.get_drug_embeddings(z)
        return self.decoder(zd[eli[0]], zd[eli[1]])
    def forward(self, ei, et, eli):
        return self.decode(self.encode(ei, et), eli)

RELATION_MAP = {
    ('drug','interacts','drug'):0,
    ('drug','targets','gene'):1, ('gene','targeted_by','drug'):2,
    ('disease','associated_with','gene'):3, ('gene','associated_with','disease'):4,
    ('disease','treated_by','drug'):5, ('drug','treats','disease'):6,
}
NODE_TYPES_ORDER = ['disease','drug','gene']

def flatten_hetero_graph(hd, ntypes_order, rmap):
    offsets = {}; off = 0
    for n in ntypes_order:
        offsets[n] = off; off += hd[n].num_nodes
    src, dst, ts = [], [], []
    for (s,r,d), rid in rmap.items():
        if (s,r,d) in hd.edge_types:
            ei = hd[(s,r,d)].edge_index
            src.append(ei[0] + offsets[s]); dst.append(ei[1] + offsets[d])
            ts.append(torch.full((ei.size(1),), rid, dtype=torch.long))
    return torch.stack([torch.cat(src), torch.cat(dst)], dim=0), torch.cat(ts), offsets

def compute_common_neighbor_scores(edge_index, num_nodes):
    src, dst = edge_index[0].numpy(), edge_index[1].numpy()
    A = coo_matrix((np.ones(len(src)), (src, dst)), shape=(num_nodes, num_nodes)).tocsr()
    M = (A @ A).toarray()
    np.fill_diagonal(M, 0)
    return M

In [ ]:
# ── Training & evaluation ────────────────────────────────────────────
def sample_negatives(ei, num_nodes, num_neg, seed=None):
    rng = np.random.RandomState(seed) if seed is not None else np.random
    pos = set(zip(ei[0].tolist(), ei[1].tolist()))
    src, dst = [], []
    while len(src) < num_neg:
        s = rng.randint(0, num_nodes); d = rng.randint(0, num_nodes)
        if s != d and (s,d) not in pos and (d,s) not in pos:
            src.append(s); dst.append(d)
    return torch.tensor([src, dst], dtype=torch.long)

def _labels(p, n, dev):
    return torch.cat([torch.ones(p), torch.zeros(n)]).to(dev)

def train_gcn(model, opt, train_ei, val_ei, graph_ei, num_nodes,
              epochs=200, patience=20, neg_ratio=5, device='cpu', verbose=True):
    best, best_state, pc = 0, None, 0
    hist = {'train_loss':[], 'val_loss':[], 'val_auroc':[]}
    val_neg = sample_negatives(val_ei, num_nodes, val_ei.shape[1]*neg_ratio, seed=999)
    val_eli = torch.cat([val_ei, val_neg], 1).to(device)
    val_lab = _labels(val_ei.shape[1], val_neg.shape[1], device)
    g_ei = graph_ei.to(device)
    pbar = tqdm(range(epochs), disable=not verbose, desc='GCN')
    for e in pbar:
        model.train()
        pos = train_ei.to(device)
        neg = sample_negatives(train_ei, num_nodes, pos.size(1)*neg_ratio).to(device)
        eli = torch.cat([pos, neg], 1)
        lab = _labels(pos.size(1), neg.size(1), device)
        opt.zero_grad()
        s = model(g_ei, eli)
        loss = F.binary_cross_entropy_with_logits(s, lab)
        loss.backward(); opt.step()
        hist['train_loss'].append(loss.item())
        model.eval()
        with torch.no_grad():
            vs = model(g_ei, val_eli)
            vl = F.binary_cross_entropy_with_logits(vs, val_lab).item()
            va = roc_auc_score(val_lab.cpu(), torch.sigmoid(vs).cpu())
        hist['val_loss'].append(vl); hist['val_auroc'].append(va)
        pbar.set_postfix(tloss=f'{loss.item():.4f}', auroc=f'{va:.4f}')
        if va > best:
            best, best_state, pc = va, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            pc += 1
            if pc >= patience: break
    if best_state: model.load_state_dict(best_state)
    return hist, best

def train_rgcn(model, opt, train_ei, val_ei, g_ei, g_et, num_drugs,
               epochs=200, patience=30, neg_ratio=5, device='cpu', verbose=True):
    best, best_state, pc = 0, None, 0
    hist = {'train_loss':[], 'val_loss':[], 'val_auroc':[]}
    val_neg = sample_negatives(val_ei, num_drugs, val_ei.shape[1]*neg_ratio, seed=999)
    val_eli = torch.cat([val_ei, val_neg], 1).to(device)
    val_lab = _labels(val_ei.shape[1], val_neg.shape[1], device)
    g_ei = g_ei.to(device); g_et = g_et.to(device)
    pbar = tqdm(range(epochs), disable=not verbose, desc='RGCN')
    for e in pbar:
        model.train()
        pos = train_ei.to(device)
        neg = sample_negatives(train_ei, num_drugs, pos.size(1)*neg_ratio).to(device)
        eli = torch.cat([pos, neg], 1)
        lab = _labels(pos.size(1), neg.size(1), device)
        opt.zero_grad()
        s = model(g_ei, g_et, eli)
        loss = F.binary_cross_entropy_with_logits(s, lab)
        loss.backward(); opt.step()
        hist['train_loss'].append(loss.item())
        model.eval()
        with torch.no_grad():
            vs = model(g_ei, g_et, val_eli)
            vl = F.binary_cross_entropy_with_logits(vs, val_lab).item()
            va = roc_auc_score(val_lab.cpu(), torch.sigmoid(vs).cpu())
        hist['val_loss'].append(vl); hist['val_auroc'].append(va)
        pbar.set_postfix(tloss=f'{loss.item():.4f}', auroc=f'{va:.4f}')
        if va > best:
            best, best_state, pc = va, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            pc += 1
            if pc >= patience: break
    if best_state: model.load_state_dict(best_state)
    return hist, best

@torch.no_grad()
def compute_link_scores(model, ei, kind='gcn', g_ei=None, g_et=None, device='cpu'):
    model.eval()
    if kind == 'gcn':
        z = model.encode(g_ei.to(device))
        s = model.decode(z, ei.to(device))
    else:
        z = model.encode(g_ei.to(device), g_et.to(device))
        s = model.decode(z, ei.to(device))
    return s.cpu()

def compute_metrics(pos_s, neg_s):
    s = torch.cat([pos_s, neg_s]).numpy()
    y = np.concatenate([np.ones(len(pos_s)), np.zeros(len(neg_s))])
    p = 1/(1+np.exp(-s))
    return {'auroc': roc_auc_score(y,p), 'auprc': average_precision_score(y,p)}

def compute_curves(pos_s, neg_s):
    s = torch.cat([pos_s, neg_s]).numpy()
    y = np.concatenate([np.ones(len(pos_s)), np.zeros(len(neg_s))])
    p = 1/(1+np.exp(-s))
    fpr, tpr, _ = roc_curve(y, p)
    pre, rec, _ = precision_recall_curve(y, p)
    return {'fpr':fpr, 'tpr':tpr, 'precision':pre, 'recall':rec}

@torch.no_grad()
def compute_ranking_metrics(model, test_pos, num_drugs, kind='gcn',
                            g_ei=None, g_et=None, device='cpu', ks=(10,20,50)):
    model.eval()
    if kind == 'gcn':
        z = model.encode(g_ei.to(device)).cpu()
    else:
        z = model.get_drug_embeddings(model.encode(g_ei.to(device), g_et.to(device))).cpu()
    all_s = z @ z.t()
    src, dst = test_pos[0].numpy(), test_pos[1].numpy()
    ranks = []
    for u, v in zip(src, dst):
        s = all_s[u].clone(); s[u] = -1e9
        ranks.append((s > s[v]).sum().item() + 1)
    ranks = np.array(ranks, dtype=np.float64)
    out = {'mrr': float((1.0/ranks).mean())}
    for k in ks: out[f'hits@{k}'] = float((ranks <= k).mean())
    return out

In [ ]:
# ── Perturbation & stability ─────────────────────────────────────────
def gaussian_noise_perturbation(z, sigma_rel, seed=None):
    if seed is not None: torch.manual_seed(seed)
    sigma = sigma_rel * z.norm(dim=1).mean().item()
    return z + torch.randn_like(z) * sigma

def dimensional_dropout_perturbation(z, rate, seed=None):
    if seed is not None: torch.manual_seed(seed)
    return z * torch.bernoulli(torch.full_like(z, 1.0 - rate))

def run_perturbation_trials(z, fn, strengths, num_trials=10, base_seed=42):
    return {s: [fn(z, s, seed=base_seed+t) for t in range(num_trials)] for s in strengths}

@torch.no_grad()
def prediction_probability_shift(z0, z1, eval_pairs=None):
    if eval_pairs is not None:
        s,d = eval_pairs
        a = torch.sigmoid((z0[s]*z0[d]).sum(-1))
        b = torch.sigmoid((z1[s]*z1[d]).sum(-1))
    else:
        a = torch.sigmoid(z0 @ z0.t())
        b = torch.sigmoid(z1 @ z1.t())
    dp = (a - b).abs()
    if eval_pairs is None:
        n = z0.size(0); dp = dp[~torch.eye(n, dtype=torch.bool)]
    return {'mean_delta_p':dp.mean().item(), 'median_delta_p':dp.median().item(),
            'max_delta_p':dp.max().item(), 'std_delta_p':dp.std().item()}

@torch.no_grad()
def ranking_shift(z0, z1):
    a = (z0 @ z0.t()).cpu().numpy(); b = (z1 @ z1.t()).cpu().numpy()
    np.fill_diagonal(a, -np.inf); np.fill_diagonal(b, -np.inf)
    rs = []
    for i in range(a.shape[0]):
        r,_ = spearmanr(a[i], b[i])
        if not np.isnan(r): rs.append(r)
    return {'mean_spearman_rho': float(np.mean(rs)), 'std_spearman_rho': float(np.std(rs))}

@torch.no_grad()
def topk_jaccard(z0, z1, k=20):
    a = (z0 @ z0.t()).clone(); b = (z1 @ z1.t()).clone()
    a.fill_diagonal_(-1e9); b.fill_diagonal_(-1e9)
    a_top = a.topk(k, dim=1).indices.cpu().numpy()
    b_top = b.topk(k, dim=1).indices.cpu().numpy()
    js = []
    for i in range(a_top.shape[0]):
        A,B = set(a_top[i]), set(b_top[i])
        js.append(len(A&B)/len(A|B))
    return {f'mean_jaccard_top{k}': float(np.mean(js)),
            f'std_jaccard_top{k}':  float(np.std(js))}

def stability_score(dp, rho, j20):
    return (max(0., 1.0 - dp) + max(0., rho) + max(0., j20)) / 3.0

def full_stability_eval(z0, z1, eval_pairs=None, ks=(10,20,50)):
    m = {}
    m.update(prediction_probability_shift(z0, z1, eval_pairs))
    m.update(ranking_shift(z0, z1))
    for k in ks:
        m.update(topk_jaccard(z0, z1, k=k))
    m['stability_score'] = stability_score(m['mean_delta_p'], m['mean_spearman_rho'], m['mean_jaccard_top20'])
    return m

def aggregate_trial_metrics(lst):
    out = {}
    for k in lst[0].keys():
        v = [m[k] for m in lst]
        out[f'{k}_mean'] = float(np.mean(v))
        out[f'{k}_std']  = float(np.std(v))
    return out

## § 5 — Data & graph constructionReproduces the four-dataset Stanford BioSNAP fusion described in the blog. Ifprocessed pickles are already cached on disk we reuse them; otherwise we buildfrom the raw TSVs.

In [ ]:
# Always load raw counts first — these are the dataset card numbers on the blog
# (and the 551,665 hero stat is their sum).
print('[load] reading raw BioSNAP files for dataset-card counts')
dfs = load_all()
raw_counts = {n: len(d) for n, d in dfs.items()}
print(f'\n=== Raw dataset counts (matches blog dataset cards) ===')
for n, c in raw_counts.items(): print(f'  {n:>14s}: {c:>8,} edges')
print(f'  {"TOTAL (hero)":>14s}: {sum(raw_counts.values()):>8,} edges')

if os.path.exists(os.path.join(DATA_PROC, 'id_maps.pkl')):
    print('\n[cached] loading processed graph from disk')
    id_maps, edges, stats = load_processed()
else:
    print('\n[build] constructing heterogeneous graph')
    ddi, dtg, dga, dda = dfs['ChCh-Miner'], dfs['ChG-Miner'], dfs['DG-AssocMiner'], dfs['DCh-Miner']
    dtg = prefix_chg_genes(dtg)
    id_maps, stats = build_id_maps(ddi, dtg, dga, dda)
    edges = build_all_edges(ddi, dtg, dga, id_maps, dda)
    save_processed(id_maps, edges, stats)

num_drugs = len(id_maps['drug'])
print(f'\n=== Processed graph (after filtering to nodes that appear in DDI) ===')
print(f'  Drugs   : {len(id_maps["drug"]):,}')
print(f'  Genes   : {len(id_maps["gene"]):,}')
print(f'  Diseases: {len(id_maps["disease"]):,}')
total = 0
for n, ei in sorted(edges.items()):
    print(f'  {n:>12s}: {ei.shape[1]:,} edges'); total += ei.shape[1]
print(f'  TOTAL: {total:,} edges')

In [ ]:
# ── Train / val / test split (80/10/10 of canonical DDI edges) ──
SPLIT_FILES = ['train_edges.pt', 'val_edges.pt', 'test_edges.pt', 'train_edges_ud.pt']
if all(os.path.exists(os.path.join(DATA_SPLITS, f)) for f in SPLIT_FILES):
    print('[cached] loading splits')
    train_edges    = torch.load(os.path.join(DATA_SPLITS, 'train_edges.pt'),    weights_only=True)
    val_edges      = torch.load(os.path.join(DATA_SPLITS, 'val_edges.pt'),      weights_only=True)
    test_edges     = torch.load(os.path.join(DATA_SPLITS, 'test_edges.pt'),     weights_only=True)
    train_edges_ud = torch.load(os.path.join(DATA_SPLITS, 'train_edges_ud.pt'), weights_only=True)
else:
    print('[build] making 80/10/10 split of DDI edges')
    set_seed(SEED)
    ddi_ei = edges['drug_drug']
    canon = ddi_ei[:, ddi_ei[0] < ddi_ei[1]]
    n = canon.shape[1]
    perm = np.random.permutation(n)
    nt, nv = int(0.8*n), int(0.1*n)
    train_e = canon[:, perm[:nt]]
    val_e   = canon[:, perm[nt:nt+nv]]
    test_e  = canon[:, perm[nt+nv:]]
    train_ud = np.concatenate([train_e, np.stack([train_e[1], train_e[0]])], axis=1)
    train_edges    = torch.from_numpy(train_e).long()
    val_edges      = torch.from_numpy(val_e).long()
    test_edges     = torch.from_numpy(test_e).long()
    train_edges_ud = torch.from_numpy(train_ud).long()
    for n_, t in zip(SPLIT_FILES, [train_edges, val_edges, test_edges, train_edges_ud]):
        torch.save(t, os.path.join(DATA_SPLITS, n_))

print(f'Train: {train_edges.shape[1]:,}  Val: {val_edges.shape[1]:,}  Test: {test_edges.shape[1]:,}')

## § 6.1 — Heuristic baseline (common neighbors)A parameter-free baseline used as the credibility floor in the blog. The scorefor a drug pair `(u, v)` is the number of training-graph common neighbors.

In [ ]:
set_seed(SEED)
cn = compute_common_neighbor_scores(train_edges_ud, num_drugs)
neg_test_h = sample_negatives(test_edges, num_drugs, test_edges.shape[1]*5, seed=42)
pos_h = cn[test_edges[0].numpy(), test_edges[1].numpy()]
neg_h = cn[neg_test_h[0].numpy(), neg_test_h[1].numpy()]
heuristic_metrics = compute_metrics(torch.tensor(pos_h, dtype=torch.float),
                                    torch.tensor(neg_h, dtype=torch.float))
heuristic_curves  = compute_curves(torch.tensor(pos_h, dtype=torch.float),
                                   torch.tensor(neg_h, dtype=torch.float))
print(f'Heuristic AUROC: {heuristic_metrics["auroc"]:.4f}')
print(f'Heuristic AUPRC: {heuristic_metrics["auprc"]:.4f}')

## § 6.2 — GCN (structure-only)2-layer GCN on the homogeneous DDI graph, dot-product decoder, BCE loss with1:5 negative sampling.

In [ ]:
GCN_CKPT = os.path.join(CKPT_DIR, f'gcn_base_s{SEED}.pt')

set_seed(SEED)
gcn = GCNLinkPredictor(num_drugs, embed_dim=64, dropout=0.3).to(DEVICE)

if os.path.exists(GCN_CKPT) and not RUN_FROM_SCRATCH:
    print(f'[cached] loading {GCN_CKPT}')
    gcn.load_state_dict(torch.load(GCN_CKPT, map_location=DEVICE, weights_only=True))
    gcn_history = json.load(open(os.path.join(RESULTS_DIR, 'gcn_history.json')))
else:
    opt = torch.optim.Adam(gcn.parameters(), lr=0.01)
    gcn_history, _ = train_gcn(gcn, opt, train_edges, val_edges, train_edges_ud,
                               num_drugs, epochs=EPOCHS_GCN, patience=20,
                               neg_ratio=5, device=DEVICE)
    torch.save(gcn.state_dict(), GCN_CKPT)
    json.dump(gcn_history, open(os.path.join(RESULTS_DIR, 'gcn_history.json'), 'w'))

In [ ]:
# ── GCN test-set evaluation ──
set_seed(SEED)
neg_test = sample_negatives(test_edges, num_drugs, test_edges.shape[1]*5, seed=42)
pos_s = compute_link_scores(gcn, test_edges, 'gcn', train_edges_ud, device=DEVICE)
neg_s = compute_link_scores(gcn, neg_test, 'gcn', train_edges_ud, device=DEVICE)
gcn_metrics = compute_metrics(pos_s, neg_s)
gcn_metrics.update(compute_ranking_metrics(gcn, test_edges, num_drugs, 'gcn',
                                           train_edges_ud, device=DEVICE))
gcn_curves = compute_curves(pos_s, neg_s)
print('GCN test metrics:')
for k,v in gcn_metrics.items(): print(f'  {k:>10s}: {v:.4f}')
json.dump(gcn_metrics, open(os.path.join(RESULTS_DIR, 'gcn_base_s42_metrics.json'), 'w'))

## § 6.3 — R-GCN (multi-relational)2-layer Relational GCN on the full heterogeneous graph (7 relation types,basis decomposition with 2 bases).

In [ ]:
RGCN_CKPT = os.path.join(CKPT_DIR, f'rgcn_base_s{SEED}.pt')

# Build the training-time hetero graph (DDI message passing uses train edges only)
hetero_data = build_hetero_data(id_maps, edges)
train_hd = HeteroData()
for n in NODE_TYPES_ORDER:
    train_hd[n].num_nodes = hetero_data[n].num_nodes
train_hd['drug', 'interacts', 'drug'].edge_index = train_edges_ud.long()
for k in hetero_data.edge_types:
    if k != ('drug','interacts','drug'):
        train_hd[k].edge_index = hetero_data[k].edge_index
train_flat_ei, train_flat_et, _ = flatten_hetero_graph(train_hd, NODE_TYPES_ORDER, RELATION_MAP)
print(f'Training graph: {train_flat_ei.shape[1]:,} edges, {len(RELATION_MAP)} relation types')

set_seed(SEED)
num_nodes_dict = {n: len(id_maps[n]) for n in NODE_TYPES_ORDER}
rgcn = RGCNLinkPredictor(num_nodes_dict, embed_dim=64,
                         num_relations=len(RELATION_MAP), num_bases=2, dropout=0.3).to(DEVICE)

if os.path.exists(RGCN_CKPT) and not RUN_FROM_SCRATCH:
    print(f'[cached] loading {RGCN_CKPT}')
    rgcn.load_state_dict(torch.load(RGCN_CKPT, map_location=DEVICE, weights_only=True))
    rgcn_history = json.load(open(os.path.join(RESULTS_DIR, 'rgcn_history.json')))
else:
    opt = torch.optim.Adam(rgcn.parameters(), lr=0.01)
    rgcn_history, _ = train_rgcn(rgcn, opt, train_edges, val_edges,
                                 train_flat_ei, train_flat_et, num_drugs,
                                 epochs=EPOCHS_RGCN, patience=50, neg_ratio=5, device=DEVICE)
    torch.save(rgcn.state_dict(), RGCN_CKPT)
    json.dump(rgcn_history, open(os.path.join(RESULTS_DIR, 'rgcn_history.json'), 'w'))

In [ ]:
# ── RGCN test-set evaluation ──
set_seed(SEED)
neg_test = sample_negatives(test_edges, num_drugs, test_edges.shape[1]*5, seed=42)
pos_s = compute_link_scores(rgcn, test_edges, 'rgcn', train_flat_ei, train_flat_et, device=DEVICE)
neg_s = compute_link_scores(rgcn, neg_test, 'rgcn', train_flat_ei, train_flat_et, device=DEVICE)
rgcn_metrics = compute_metrics(pos_s, neg_s)
rgcn_metrics.update(compute_ranking_metrics(rgcn, test_edges, num_drugs, 'rgcn',
                                            train_flat_ei, train_flat_et, device=DEVICE))
rgcn_curves = compute_curves(pos_s, neg_s)
print('RGCN test metrics:')
for k,v in rgcn_metrics.items(): print(f'  {k:>10s}: {v:.4f}')
json.dump(rgcn_metrics, open(os.path.join(RESULTS_DIR, 'rgcn_base_s42_metrics.json'), 'w'))

## § 7 — Predictive accuracy figuresThese render the matplotlib equivalents of the Chart.js panels in §7 of theblog: model-accuracy bars, Hits@K bars, and the approximate ROC curves.

In [ ]:
# ── Figure: AUROC + AUPRC bars (mirror of blog 'chart-accuracy') ──
labels = ['Heuristic', 'GCN', 'RGCN']
auroc_vals = [heuristic_metrics['auroc'], gcn_metrics['auroc'], rgcn_metrics['auroc']]
auprc_vals = [heuristic_metrics['auprc'], gcn_metrics['auprc'], rgcn_metrics['auprc']]

fig, ax = plt.subplots(figsize=(8,4.5))
x = np.arange(len(labels)); w = 0.35
b1 = ax.bar(x - w/2, auroc_vals, w, label='AUROC',
            color=[COLORS['heuristic'], COLORS['gcn'], COLORS['rgcn']])
b2 = ax.bar(x + w/2, auprc_vals, w, label='AUPRC',
            color=[COLORS_LITE['heuristic'], COLORS_LITE['gcn'], COLORS_LITE['rgcn']])
for b, v in zip(b1, auroc_vals): ax.text(b.get_x()+b.get_width()/2, v+.01, f'{v:.3f}', ha='center', fontsize=10)
for b, v in zip(b2, auprc_vals): ax.text(b.get_x()+b.get_width()/2, v+.01, f'{v:.3f}', ha='center', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('Score'); ax.set_ylim(0.35, 1.0)
ax.set_title('Figure · Model accuracy — AUROC & AUPRC')
ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(BLOG_ASSETS, 'fig1_accuracy.png'), dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ── Figure: Hits@K bars ──
ks = [10, 20, 50]
gcn_hits  = [gcn_metrics[f'hits@{k}']  for k in ks]
rgcn_hits = [rgcn_metrics[f'hits@{k}'] for k in ks]

fig, ax = plt.subplots(figsize=(8,4.5))
x = np.arange(len(ks)); w = 0.35
ax.bar(x - w/2, gcn_hits,  w, label='GCN',  color=COLORS['gcn'])
ax.bar(x + w/2, rgcn_hits, w, label='RGCN', color=COLORS['rgcn'])
for i,v in enumerate(gcn_hits):  ax.text(i - w/2, v+.005, f'{v:.3f}', ha='center', fontsize=10)
for i,v in enumerate(rgcn_hits): ax.text(i + w/2, v+.005, f'{v:.3f}', ha='center', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels([f'Hits@{k}' for k in ks])
ax.set_ylabel('Hit rate'); ax.set_ylim(0, max(rgcn_hits)*1.25)
ax.set_title('Figure · Ranking performance — Hits@K')
ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure: ROC curves (heuristic + GCN + RGCN) ──
fig, ax = plt.subplots(figsize=(8,5.5))
ax.plot([0,1],[0,1], 'k--', alpha=.3, label='Random (0.500)')
ax.plot(heuristic_curves['fpr'], heuristic_curves['tpr'],
        color=COLORS['heuristic'], lw=2, label=f'Heuristic ({heuristic_metrics["auroc"]:.3f})')
ax.plot(gcn_curves['fpr'], gcn_curves['tpr'],
        color=COLORS['gcn'], lw=2, label=f'GCN ({gcn_metrics["auroc"]:.3f})')
ax.plot(rgcn_curves['fpr'], rgcn_curves['tpr'],
        color=COLORS['rgcn'], lw=2.2, label=f'RGCN ({rgcn_metrics["auroc"]:.3f})')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('Figure · ROC curves')
ax.legend(loc='lower right'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## § 7 — Training curves (Figure 3a/3b in the blog)The dashed lines are validation; solid lines are training.

In [ ]:
# ── Figure 3a · loss curves; 3b · val AUROC curves ──
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
ge = np.arange(len(gcn_history['train_loss'])) / max(1, len(gcn_history['train_loss'])-1) * 100
re = np.arange(len(rgcn_history['train_loss'])) / max(1, len(rgcn_history['train_loss'])-1) * 100

a.plot(ge, gcn_history['train_loss'],  color=COLORS['gcn'],  lw=2,   label='GCN train')
a.plot(ge, gcn_history['val_loss'],    color=COLORS_LITE['gcn'],  lw=1.5, ls='--', label='GCN val')
a.plot(re, rgcn_history['train_loss'], color=COLORS['rgcn'], lw=2,   label='RGCN train')
a.plot(re, rgcn_history['val_loss'],   color=COLORS_LITE['rgcn'], lw=1.5, ls='--', label='RGCN val')
a.set_xlabel('% of training run'); a.set_ylabel('Binary cross-entropy')
a.set_title('Figure 3a · Training loss vs epoch'); a.grid(alpha=.3); a.legend()

b.plot(ge, gcn_history['val_auroc'],  color=COLORS['gcn'],  lw=2.4, label='GCN val AUROC')
b.plot(re, rgcn_history['val_auroc'], color=COLORS['rgcn'], lw=2.4, label='RGCN val AUROC')
b.set_xlabel('% of training run'); b.set_ylabel('Validation AUROC')
b.set_title('Figure 3b · Validation AUROC vs epoch'); b.grid(alpha=.3); b.legend()

plt.tight_layout(); plt.show()

## § 8 — Stability under Gaussian noiseWe extract the learned drug embeddings from each frozen model and injectisotropic Gaussian noise scaled to the mean embedding norm. Each σ is repeated`NUM_TRIALS` times with independent seeds. The Stability Score is theequally-weighted average of (1 − mean|Δp|), Spearman ρ, and top-20 Jaccard.

In [ ]:
with torch.no_grad():
    gcn_z  = gcn.encode(train_edges_ud.to(DEVICE)).cpu()
    rgcn_z = rgcn.get_drug_embeddings(rgcn.encode(train_flat_ei.to(DEVICE), train_flat_et.to(DEVICE))).cpu()
print(f'GCN  drug embeddings: {tuple(gcn_z.shape)}, mean ‖z‖ = {gcn_z.norm(dim=1).mean():.4f}')
print(f'RGCN drug embeddings: {tuple(rgcn_z.shape)}, mean ‖z‖ = {rgcn_z.norm(dim=1).mean():.4f}')

In [ ]:
NOISE_LEVELS = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]
gauss_results = {}
for name, z in [('gcn', gcn_z), ('rgcn', rgcn_z)]:
    print(f'=== {name.upper()} Gaussian noise ===')
    trials = run_perturbation_trials(z, gaussian_noise_perturbation, NOISE_LEVELS, NUM_TRIALS)
    res = {}
    for s in NOISE_LEVELS:
        tm = [full_stability_eval(z, zp) for zp in trials[s]]
        agg = aggregate_trial_metrics(tm)
        res[s] = agg
        print(f'  σ={s:.2f}: SS={agg["stability_score_mean"]:.4f}±{agg["stability_score_std"]:.4f}'
              f'  ρ={agg["mean_spearman_rho_mean"]:.4f}'
              f'  J@20={agg["mean_jaccard_top20_mean"]:.4f}')
    gauss_results[name] = res

In [ ]:
# ── Figure: Stability Score vs σ (mirrors blog 'chart-ss') ──
fig, ax = plt.subplots(figsize=(9,5))
for name in ('gcn', 'rgcn'):
    res = gauss_results[name]
    s = sorted(res.keys()); xs = np.array(s)
    m = np.array([res[k]['stability_score_mean'] for k in s])
    sd = np.array([res[k]['stability_score_std']  for k in s])
    ax.plot(xs, m, 'o-', color=COLORS[name], lw=2, label=name.upper())
    ax.fill_between(xs, m-sd, m+sd, color=COLORS[name], alpha=.15)
ax.set_xlabel('Gaussian noise σ (relative to mean ‖z‖)')
ax.set_ylabel('Stability Score'); ax.set_ylim(0.3, 1.02)
ax.set_title('Figure · Stability Score vs Gaussian noise (10 trials per σ)')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(BLOG_ASSETS, 'fig2_stability.png'), dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ── Figure: Spearman ρ (chart-rho) ──
fig, ax = plt.subplots(figsize=(9,4.5))
for name in ('gcn', 'rgcn'):
    res = gauss_results[name]; s = sorted(res.keys())
    ax.plot(s, [res[k]['mean_spearman_rho_mean'] for k in s], 'o-',
            color=COLORS[name], lw=2, label=f'{name.upper()} Spearman ρ')
ax.set_xlabel('Gaussian noise σ'); ax.set_ylabel('Spearman ρ')
ax.set_title('Figure · Ranking correlation under Gaussian noise')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure: mean |Δp| (chart-deltap) ──
fig, ax = plt.subplots(figsize=(9,4.5))
for name in ('gcn', 'rgcn'):
    res = gauss_results[name]; s = sorted(res.keys())
    ax.plot(s, [res[k]['mean_delta_p_mean'] for k in s], 'o-',
            color=COLORS[name], lw=2, label=f'{name.upper()} mean |Δp|')
ax.set_xlabel('Gaussian noise σ'); ax.set_ylabel('Mean |Δp|')
ax.set_title('Figure · Predicted-probability shift under Gaussian noise')
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=.3, which='both')
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure: top-K Jaccard (chart-jaccard) ──
fig, ax = plt.subplots(figsize=(10, 5))
styles = {10: '-', 20: '--', 50: ':'}
for k, ls in styles.items():
    for name in ('gcn', 'rgcn'):
        res = gauss_results[name]; s = sorted(res.keys())
        ax.plot(s, [res[x][f'mean_jaccard_top{k}_mean'] for x in s], ls,
                color=COLORS[name], lw=2, label=f'{name.upper()} K={k}')
ax.set_xlabel('Gaussian noise σ'); ax.set_ylabel('Top-K Jaccard overlap')
ax.set_ylim(0, 1.0); ax.set_title('Figure · Top-K Jaccard overlap under Gaussian noise')
ax.legend(ncol=2); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(BLOG_ASSETS, 'fig3_topk.png'), dpi=300, bbox_inches='tight'); plt.show()

## § 9 — Stability under dimensional dropoutWe zero out a random subset of embedding dimensions per trial.

In [ ]:
DROPOUT_RATES = [0.05, 0.10, 0.20, 0.30, 0.50]
dropout_results = {}
for name, z in [('gcn', gcn_z), ('rgcn', rgcn_z)]:
    print(f'=== {name.upper()} dimensional dropout ===')
    trials = run_perturbation_trials(z, dimensional_dropout_perturbation, DROPOUT_RATES, NUM_TRIALS)
    res = {}
    for r in DROPOUT_RATES:
        tm = [full_stability_eval(z, zp) for zp in trials[r]]
        agg = aggregate_trial_metrics(tm); res[r] = agg
        print(f'  rate={r:.2f}: SS={agg["stability_score_mean"]:.4f}'
              f'  ρ={agg["mean_spearman_rho_mean"]:.4f}')
    dropout_results[name] = res

In [ ]:
# ── Figure: dropout SS (chart-dss) and dropout ρ (chart-drho) ──
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
for name in ('gcn', 'rgcn'):
    res = dropout_results[name]; rs = sorted(res.keys())
    a.plot(rs, [res[r]['stability_score_mean'] for r in rs], 'o-',
           color=COLORS[name], lw=2, label=name.upper())
    b.plot(rs, [res[r]['mean_spearman_rho_mean'] for r in rs], 'o-',
           color=COLORS[name], lw=2, label=f'{name.upper()} Spearman ρ')
a.set_xlabel('Dropout rate'); a.set_ylabel('Stability Score')
a.set_title('Figure · SS vs dimensional dropout'); a.grid(alpha=.3); a.legend()
b.set_xlabel('Dropout rate'); b.set_ylabel('Spearman ρ')
b.set_title('Figure · Spearman ρ vs dimensional dropout'); b.grid(alpha=.3); b.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure: head-to-head Gaussian vs dropout (chart-compare) ──
levels = [('Level 1\n(low)', 0.01, 0.05),
          ('Level 2',          0.05, 0.10),
          ('Level 3\n(mod.)', 0.10, 0.20),
          ('Level 4',          0.15, 0.30),
          ('Level 5\n(high)', 0.20, 0.50)]
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(levels)); w = 0.18
gc_g = [gauss_results['gcn'][s]['stability_score_mean']  for _,s,_ in levels]
gc_d = [dropout_results['gcn'][r]['stability_score_mean'] for _,_,r in levels]
rg_g = [gauss_results['rgcn'][s]['stability_score_mean']  for _,s,_ in levels]
rg_d = [dropout_results['rgcn'][r]['stability_score_mean'] for _,_,r in levels]
ax.bar(x - 1.5*w, gc_g, w, color=COLORS['gcn'],     alpha=.85, label='GCN — Gaussian')
ax.bar(x - 0.5*w, gc_d, w, color=COLORS['gcn'],     alpha=.45, label='GCN — Dropout')
ax.bar(x + 0.5*w, rg_g, w, color=COLORS['rgcn'],    alpha=.85, label='RGCN — Gaussian')
ax.bar(x + 1.5*w, rg_d, w, color=COLORS['rgcn'],    alpha=.45, label='RGCN — Dropout')
ax.set_xticks(x); ax.set_xticklabels([l for l,_,_ in levels])
ax.set_ylabel('Stability Score'); ax.set_ylim(0.4, 1.0)
ax.set_title('Figure · Gaussian vs dropout — matched-severity Stability Score')
ax.legend(ncol=2); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure 5a · SS components for the RGCN ──
fig, ax = plt.subplots(figsize=(9, 4.5))
sigs = sorted(gauss_results['rgcn'].keys())
inv_dp = [1.0 - min(1.0, gauss_results['rgcn'][s]['mean_delta_p_mean']) for s in sigs]
rho    = [gauss_results['rgcn'][s]['mean_spearman_rho_mean']            for s in sigs]
j20    = [gauss_results['rgcn'][s]['mean_jaccard_top20_mean']           for s in sigs]
ss     = [gauss_results['rgcn'][s]['stability_score_mean']              for s in sigs]
ax.plot(sigs, inv_dp, 'o-', color='#f59e0b', lw=2, label='1 − mean|Δp|')
ax.plot(sigs, rho,    'o-', color=COLORS['rgcn'], lw=2, label='Spearman ρ')
ax.plot(sigs, j20,    'o-', color='#10b981', lw=2, label='Top-20 Jaccard')
ax.plot(sigs, ss,     'o--', color='#0f172a', lw=2.4, label='Composite SS')
ax.set_xlabel('Gaussian noise σ'); ax.set_ylabel('Component value')
ax.set_ylim(0, 1.05); ax.set_title('Figure 5a · SS components (RGCN)')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure 5b · per-drug top-1 partner flip rate ──
@torch.no_grad()
def top1_flip_rate(z0, z1):
    a = z0 @ z0.t(); b = z1 @ z1.t()
    a.fill_diagonal_(-1e9); b.fill_diagonal_(-1e9)
    a1 = a.argmax(dim=1).cpu().numpy()
    b1 = b.argmax(dim=1).cpu().numpy()
    return float((a1 != b1).mean())

flip = {'gcn': [], 'rgcn': []}
for name, z in [('gcn', gcn_z), ('rgcn', rgcn_z)]:
    for s in NOISE_LEVELS:
        rates = []
        for t in range(NUM_TRIALS):
            zp = gaussian_noise_perturbation(z, s, seed=42+t)
            rates.append(top1_flip_rate(z, zp))
        flip[name].append(float(np.mean(rates)))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(NOISE_LEVELS, flip['gcn'],  'o-', color=COLORS['gcn'],  lw=2, label='GCN')
ax.plot(NOISE_LEVELS, flip['rgcn'], 'o-', color=COLORS['rgcn'], lw=2, label='RGCN')
ax.set_xlabel('Gaussian noise σ'); ax.set_ylabel('Fraction of drugs whose #1 partner changes')
ax.set_ylim(0, 1.0); ax.set_title('Figure 5b · Per-drug top-1 partner flip rate')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Persist stability JSON for downstream consumers
def _strkey(d): return {str(k): v for k, v in d.items()}
stab_data = {'gaussian':  {m: _strkey(r) for m, r in gauss_results.items()},
             'dropout':   {m: _strkey(r) for m, r in dropout_results.items()}}
json.dump(stab_data, open(os.path.join(RESULTS_DIR, 'stability_results.json'), 'w'), indent=2)
print(f'Saved stability_results.json')

## § 10 — Edge-type ablationThree R-GCN variants, each missing one auxiliary edge type. Trains fromscratch when `RUN_FROM_SCRATCH=True`; otherwise loads cached results.

In [ ]:
ABL_PATH = os.path.join(RESULTS_DIR, 'ablation_results.json')

EDGE_GROUPS = {
    'no_drug_gene':    [('drug','targets','gene'),         ('gene','targeted_by','drug')],
    'no_disease_gene': [('disease','associated_with','gene'),('gene','associated_with','disease')],
    'no_disease_drug': [('disease','treated_by','drug'),   ('drug','treats','disease')],
}

def run_one_ablation(excluded):
    set_seed(SEED)
    abl_hd = HeteroData()
    for n in NODE_TYPES_ORDER: abl_hd[n].num_nodes = hetero_data[n].num_nodes
    abl_hd['drug','interacts','drug'].edge_index = train_edges_ud.long()
    for k in hetero_data.edge_types:
        if k != ('drug','interacts','drug') and k not in excluded:
            abl_hd[k].edge_index = hetero_data[k].edge_index
    a_ei, a_et, _ = flatten_hetero_graph(abl_hd, NODE_TYPES_ORDER, RELATION_MAP)
    m = RGCNLinkPredictor(num_nodes_dict, embed_dim=64,
                          num_relations=len(RELATION_MAP), num_bases=2, dropout=0.3).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    train_rgcn(m, opt, train_edges, val_edges, a_ei, a_et, num_drugs,
               epochs=EPOCHS_RGCN, patience=50, neg_ratio=5, device=DEVICE, verbose=False)
    nt = sample_negatives(test_edges, num_drugs, test_edges.shape[1]*5, seed=42)
    p_s = compute_link_scores(m, test_edges, 'rgcn', a_ei, a_et, device=DEVICE)
    n_s = compute_link_scores(m, nt,        'rgcn', a_ei, a_et, device=DEVICE)
    met = compute_metrics(p_s, n_s)
    with torch.no_grad():
        zf = m.encode(a_ei.to(DEVICE), a_et.to(DEVICE))
        zd = m.get_drug_embeddings(zf).cpu()
    tr = run_perturbation_trials(zd, gaussian_noise_perturbation, [0.10], NUM_TRIALS)
    s10 = aggregate_trial_metrics([full_stability_eval(zd, zp) for zp in tr[0.10]])
    met['stability_score_01'] = s10['stability_score_mean']
    return met

if os.path.exists(ABL_PATH) and not RUN_FROM_SCRATCH:
    print(f'[cached] {ABL_PATH}')
    ablation = json.load(open(ABL_PATH))
else:
    print('[run] training 3 ablation variants')
    ablation = {'full_rgcn': {**rgcn_metrics,
                              'stability_score_01': gauss_results['rgcn'][0.10]['stability_score_mean']}}
    for n, ex in EDGE_GROUPS.items():
        print(f'  variant: {n}')
        ablation[n] = run_one_ablation(ex)
    json.dump(ablation, open(ABL_PATH, 'w'), indent=2)

for k, v in ablation.items():
    print(f'  {k:>16s}: AUROC={v["auroc"]:.4f}  AUPRC={v["auprc"]:.4f}  SS@σ=.10={v["stability_score_01"]:.4f}')

In [ ]:
# ── Figure: Ablation AUROC + AUPRC (chart-abl-auroc) ──
order = ['full_rgcn', 'no_drug_gene', 'no_disease_gene', 'no_disease_drug']
labels = ['Full RGCN', 'No Drug-Gene', 'No Disease-Gene', 'No Disease-Drug']
auroc_vals = [ablation[o]['auroc'] for o in order]
auprc_vals = [ablation[o]['auprc'] for o in order]

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
xs = np.arange(len(order)); w = 0.35
a.bar(xs - w/2, auroc_vals, w, color=COLORS['rgcn'], label='AUROC')
a.bar(xs + w/2, auprc_vals, w, color=COLORS['gcn'],  label='AUPRC')
for i, v in enumerate(auroc_vals): a.text(i - w/2, v+.005, f'{v:.3f}', ha='center', fontsize=9)
for i, v in enumerate(auprc_vals): a.text(i + w/2, v+.005, f'{v:.3f}', ha='center', fontsize=9)
a.set_xticks(xs); a.set_xticklabels(labels, rotation=12, ha='right')
a.set_ylim(0.65, 0.95); a.set_title('Figure · Ablation AUROC / AUPRC')
a.legend(); a.grid(axis='y', alpha=.3)

ss_vals = [ablation[o]['stability_score_01'] for o in order]
b.bar(xs, ss_vals, color='#10b981', alpha=.75)
for i, v in enumerate(ss_vals): b.text(i, v+.001, f'{v:.3f}', ha='center', fontsize=9)
b.set_xticks(xs); b.set_xticklabels(labels, rotation=12, ha='right')
b.set_ylim(0.68, 0.77); b.set_title('Figure · Ablation Stability Score (σ=0.10)')
b.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(BLOG_ASSETS, 'fig4_ablation.png'), dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ── Figure: dual-axis accuracy / stability ──
fig, ax1 = plt.subplots(figsize=(10, 5))
xs = np.arange(len(order)); w = 0.35
ax1.bar(xs - w/2, auroc_vals, w, color=COLORS['rgcn'], label='AUROC')
ax1.set_ylabel('AUROC'); ax1.set_ylim(0.885, 0.925)
ax1.set_xticks(xs); ax1.set_xticklabels(labels, rotation=12, ha='right')
ax2 = ax1.twinx()
ax2.bar(xs + w/2, ss_vals, w, color='#10b981', alpha=.85, label='Stability Score (σ=0.10)')
ax2.set_ylabel('Stability Score (σ=0.10)'); ax2.set_ylim(0.685, 0.76)
ax1.set_title('Figure · Accuracy–stability trade-off per ablation (dual axis)')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='upper left')
ax1.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

## § 11 — Cold-start evaluationWe hold out the DDI edges of ~10% of mid-degree drugs entirely and re-train.

In [ ]:
COLD_PATH = os.path.join(RESULTS_DIR, 'coldstart_results.json')
COLD_FILES = ['cold_start_drugs.pt', 'cold_edges.pt', 'cs_train_edges.pt',
              'cs_val_edges.pt', 'cs_test_warm_edges.pt']

# Build cold-start splits if missing
if not all(os.path.exists(os.path.join(DATA_SPLITS, f)) for f in COLD_FILES):
    set_seed(SEED)
    ddi_ei = edges['drug_drug']
    canon = ddi_ei[:, ddi_ei[0] < ddi_ei[1]]
    deg = np.zeros(num_drugs, dtype=int)
    for s in ddi_ei[0]: deg[s] += 1
    deg = deg // 2
    p30 = np.percentile(deg[deg>0], 30); p60 = np.percentile(deg[deg>0], 60)
    cands = np.where((deg>=p30)&(deg<=p60))[0]
    np.random.shuffle(cands)
    n_cold = max(1, int(0.10*num_drugs))
    cold_drugs_set = set(cands[:n_cold].tolist())
    cm = np.array([s in cold_drugs_set or d in cold_drugs_set for s,d in zip(canon[0], canon[1])])
    cold = canon[:, cm]
    warm = canon[:, ~cm]
    perm = np.random.permutation(warm.shape[1])
    nt, nv = int(0.8*warm.shape[1]), int(0.1*warm.shape[1])
    cs_tr = warm[:, perm[:nt]]; cs_v = warm[:, perm[nt:nt+nv]]; cs_te = warm[:, perm[nt+nv:]]
    torch.save(torch.tensor(sorted(cold_drugs_set)), os.path.join(DATA_SPLITS, 'cold_start_drugs.pt'))
    torch.save(torch.from_numpy(cold).long(),       os.path.join(DATA_SPLITS, 'cold_edges.pt'))
    torch.save(torch.from_numpy(cs_tr).long(),      os.path.join(DATA_SPLITS, 'cs_train_edges.pt'))
    torch.save(torch.from_numpy(cs_v).long(),       os.path.join(DATA_SPLITS, 'cs_val_edges.pt'))
    torch.save(torch.from_numpy(cs_te).long(),      os.path.join(DATA_SPLITS, 'cs_test_warm_edges.pt'))

cold_drugs = torch.load(os.path.join(DATA_SPLITS, 'cold_start_drugs.pt'), weights_only=True)
cold_edges = torch.load(os.path.join(DATA_SPLITS, 'cold_edges.pt'),       weights_only=True)
cs_train   = torch.load(os.path.join(DATA_SPLITS, 'cs_train_edges.pt'),   weights_only=True)
cs_val     = torch.load(os.path.join(DATA_SPLITS, 'cs_val_edges.pt'),     weights_only=True)
cs_train_ud = torch.cat([cs_train, torch.stack([cs_train[1], cs_train[0]])], dim=1)
print(f'Cold-start: {len(cold_drugs)} drugs, {cold_edges.shape[1]} held-out edges')

In [ ]:
def run_coldstart():
    # GCN cold-start
    set_seed(SEED)
    cs_gcn = GCNLinkPredictor(num_drugs, embed_dim=64, dropout=0.3).to(DEVICE)
    train_gcn(cs_gcn, torch.optim.Adam(cs_gcn.parameters(), lr=0.001),
              cs_train, cs_val, cs_train_ud, num_drugs,
              epochs=300, patience=30, neg_ratio=5, device=DEVICE, verbose=False)
    cn = sample_negatives(cold_edges, num_drugs, cold_edges.shape[1]*5, seed=777)
    p = compute_link_scores(cs_gcn, cold_edges, 'gcn', cs_train_ud, device=DEVICE)
    n = compute_link_scores(cs_gcn, cn,         'gcn', cs_train_ud, device=DEVICE)
    g_met = compute_metrics(p, n)
    with torch.no_grad(): zg = cs_gcn.encode(cs_train_ud.to(DEVICE)).cpu()
    tr = run_perturbation_trials(zg, gaussian_noise_perturbation, [0.10], NUM_TRIALS)
    s10 = aggregate_trial_metrics([full_stability_eval(zg, zp) for zp in tr[0.10]])
    g_met['stability_score_01'] = s10['stability_score_mean']

    # RGCN cold-start
    set_seed(SEED)
    cs_hd = HeteroData()
    for nm in NODE_TYPES_ORDER: cs_hd[nm].num_nodes = hetero_data[nm].num_nodes
    cs_hd['drug','interacts','drug'].edge_index = cs_train_ud.long()
    for k in hetero_data.edge_types:
        if k != ('drug','interacts','drug'): cs_hd[k].edge_index = hetero_data[k].edge_index
    cs_ei, cs_et, _ = flatten_hetero_graph(cs_hd, NODE_TYPES_ORDER, RELATION_MAP)
    cs_rgcn = RGCNLinkPredictor(num_nodes_dict, embed_dim=64,
                                num_relations=len(RELATION_MAP), num_bases=2, dropout=0.3).to(DEVICE)
    train_rgcn(cs_rgcn, torch.optim.Adam(cs_rgcn.parameters(), lr=0.01),
               cs_train, cs_val, cs_ei, cs_et, num_drugs,
               epochs=EPOCHS_RGCN, patience=50, neg_ratio=5, device=DEVICE, verbose=False)
    p = compute_link_scores(cs_rgcn, cold_edges, 'rgcn', cs_ei, cs_et, device=DEVICE)
    n = compute_link_scores(cs_rgcn, cn,          'rgcn', cs_ei, cs_et, device=DEVICE)
    r_met = compute_metrics(p, n)
    with torch.no_grad():
        zr = cs_rgcn.get_drug_embeddings(cs_rgcn.encode(cs_ei.to(DEVICE), cs_et.to(DEVICE))).cpu()
    tr = run_perturbation_trials(zr, gaussian_noise_perturbation, [0.10], NUM_TRIALS)
    s10 = aggregate_trial_metrics([full_stability_eval(zr, zp) for zp in tr[0.10]])
    r_met['stability_score_01'] = s10['stability_score_mean']
    return {'gcn_cold': g_met, 'rgcn_cold': r_met}

if os.path.exists(COLD_PATH) and not RUN_FROM_SCRATCH:
    print(f'[cached] {COLD_PATH}')
    coldstart = json.load(open(COLD_PATH))
else:
    print('[run] training cold-start GCN + RGCN')
    coldstart = run_coldstart()
    json.dump(coldstart, open(COLD_PATH, 'w'), indent=2)

for k, v in coldstart.items():
    print(f'  {k:>10s}: AUROC={v["auroc"]:.4f}  SS@σ=.10={v["stability_score_01"]:.4f}')

In [ ]:
# ── Figure: cold-start comparison (chart-cold-auroc) ──
labels = ['GCN\nStandard', 'GCN\nCold-Start', 'RGCN\nStandard', 'RGCN\nCold-Start']
auroc_v = [gcn_metrics['auroc'], coldstart['gcn_cold']['auroc'],
           rgcn_metrics['auroc'], coldstart['rgcn_cold']['auroc']]
ss_v    = [gauss_results['gcn'][0.10]['stability_score_mean'],
           coldstart['gcn_cold']['stability_score_01'],
           gauss_results['rgcn'][0.10]['stability_score_mean'],
           coldstart['rgcn_cold']['stability_score_01']]
fig, ax1 = plt.subplots(figsize=(10, 5))
xs = np.arange(len(labels)); w = 0.35
bar_colors = [COLORS['gcn'], '#ef4444', COLORS['rgcn'], '#f59e0b']
ax1.bar(xs - w/2, auroc_v, w, color=bar_colors, label='AUROC')
for i,v in enumerate(auroc_v): ax1.text(i - w/2, v+.01, f'{v:.3f}', ha='center', fontsize=9)
ax1.set_ylabel('AUROC'); ax1.set_ylim(0, 1.05)
ax1.set_xticks(xs); ax1.set_xticklabels(labels)
ax2 = ax1.twinx()
ax2.bar(xs + w/2, ss_v, w, color=[c+'80' for c in bar_colors], label='Stability Score (σ=0.10)')
for i,v in enumerate(ss_v): ax2.text(i + w/2, v+.005, f'{v:.3f}', ha='center', fontsize=9)
ax2.set_ylabel('Stability Score'); ax2.set_ylim(0.5, 0.85)
ax1.set_title('Figure · Cold-start vs standard — AUROC collapse, SS persistence')
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='upper right')
ax1.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(BLOG_ASSETS, 'fig5_coldstart.png'), dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ── Figure: scatter — AUROC vs SS across all conditions (chart-scatter) ──
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.scatter([gcn_metrics['auroc']],  [gauss_results['gcn'][0.10]['stability_score_mean']],
           s=180, color=COLORS['gcn'], edgecolor='k', label='GCN (Std)', zorder=5)
ax.scatter([rgcn_metrics['auroc']], [gauss_results['rgcn'][0.10]['stability_score_mean']],
           s=180, color=COLORS['rgcn'], edgecolor='k', label='RGCN (Std)', zorder=5)
ax.scatter([coldstart['gcn_cold']['auroc']],  [coldstart['gcn_cold']['stability_score_01']],
           s=180, color='#ef4444', marker='^', edgecolor='k', label='GCN (Cold)', zorder=5)
ax.scatter([coldstart['rgcn_cold']['auroc']], [coldstart['rgcn_cold']['stability_score_01']],
           s=180, color='#f59e0b', marker='^', edgecolor='k', label='RGCN (Cold)', zorder=5)
abl_to_show = ['no_drug_gene', 'no_disease_gene', 'no_disease_drug']
ax.scatter([ablation[a]['auroc'] for a in abl_to_show],
           [ablation[a]['stability_score_01'] for a in abl_to_show],
           s=140, color='#10b981', marker='s', edgecolor='k', label='Ablation', zorder=5)
ax.set_xlim(0, 1); ax.set_ylim(0.5, 0.85)
ax.set_xlabel('AUROC'); ax.set_ylabel('Stability Score (σ=0.10)')
ax.set_title('Figure · AUROC vs Stability Score across all conditions')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## § 13 — Summary table & sanity check vs the blogThe numbers printed below are the ones quoted in the blog hero, thefindings cards, and the metrics table.

In [ ]:
g_g = gauss_results['gcn']; g_r = gauss_results['rgcn']
summary = pd.DataFrame({
    'Model':      ['Heuristic', 'GCN', 'RGCN'],
    'AUROC':      [heuristic_metrics['auroc'], gcn_metrics['auroc'], rgcn_metrics['auroc']],
    'AUPRC':      [heuristic_metrics['auprc'], gcn_metrics['auprc'], rgcn_metrics['auprc']],
    'MRR':        ['—', f'{gcn_metrics["mrr"]:.3f}', f'{rgcn_metrics["mrr"]:.3f}'],
    'Hits@10':    ['—', f'{gcn_metrics["hits@10"]:.3f}', f'{rgcn_metrics["hits@10"]:.3f}'],
    'Hits@20':    ['—', f'{gcn_metrics["hits@20"]:.3f}', f'{rgcn_metrics["hits@20"]:.3f}'],
    'Hits@50':    ['—', f'{gcn_metrics["hits@50"]:.3f}', f'{rgcn_metrics["hits@50"]:.3f}'],
    'SS σ=0.05':  ['—', f'{g_g[0.05]["stability_score_mean"]:.3f}', f'{g_r[0.05]["stability_score_mean"]:.3f}'],
    'SS σ=0.10':  ['—', f'{g_g[0.10]["stability_score_mean"]:.3f}', f'{g_r[0.10]["stability_score_mean"]:.3f}'],
    'SS σ=0.20':  ['—', f'{g_g[0.20]["stability_score_mean"]:.3f}', f'{g_r[0.20]["stability_score_mean"]:.3f}'],
})
try:
    from IPython.display import display
    display(summary)
except Exception:
    print(summary.to_string(index=False))
summary.to_csv(os.path.join(RESULTS_DIR, 'summary.csv'), index=False)

In [ ]:
# ── Hero stats reproduced from this notebook ──
processed_edges = sum(ei.shape[1] for ei in edges.values())
hero_edges = sum(raw_counts.values())  # the 551,665 number on the blog hero
print('=== Blog hero stats ===')
print(f'  RGCN AUROC               : {rgcn_metrics["auroc"]:.3f}    (blog: 0.917)')
print(f'  Total graph edges (raw)  : {hero_edges:,}+    (blog hero: 551,665+)')
print(f'  Total graph edges (used) : {processed_edges:,}    (post-filter to DDI drugs)')
print(f'  Stability Score σ=0.05   : {g_r[0.05]["stability_score_mean"]:.3f}    (blog: 0.833)')
print(f'  Trials per noise level   : {NUM_TRIALS}        (blog: 10×)')
print()
print('=== Blog headline numbers — cross-check ===')
print(f'  Heuristic AUROC : {heuristic_metrics["auroc"]:.3f} (blog: 0.887)')
print(f'  GCN  AUROC      : {gcn_metrics["auroc"]:.3f} (blog: 0.827)')
print(f'  RGCN AUROC      : {rgcn_metrics["auroc"]:.3f} (blog: 0.917)')
print(f'  GCN  SS σ=0.30  : {g_g[0.30]["stability_score_mean"]:.3f} (blog: 0.470)')
print(f'  RGCN SS σ=0.30  : {g_r[0.30]["stability_score_mean"]:.3f} (blog: 0.390)')

## § DoneEvery figure rendered above is reproducible from this single notebook. ThePNGs are saved to `blog/assets/fig{1..5}_*.png`, the JSON outputs go to`experiments/results/*.json`, and the summary table is saved as`experiments/results/summary.csv`.To regenerate everything from scratch (no caches), set `RUN_FROM_SCRATCH = True`in the first cell and run all again.